Векторизуйте 4 текста на одном языке (каждый ≥2500 символов). Можете использовать любой из известных вам способов. Посчитайте, какие тексты ближе друг к другу.
Положите в классрум эти 4 файла и тетрадку с кодом. Прямо в тетрадке в текстовой ячейке прокомментируйте, ожидали ли вы такого результата, почему он получился именно таким, а не другим.


In [3]:
with open("text1.txt", "r", encoding="utf-8") as f:
    text1 = f.read()
print("Количество символов:", len(text1))

with open("text2.txt", "r", encoding="utf-8") as f:
    text2 = f.read()
print("Количество символов:", len(text2))

with open("text3.txt", "r", encoding="utf-8") as f:
    text3 = f.read()
print("Количество символов:", len(text3))

with open("text4.txt", "r", encoding="utf-8") as f:
    text4 = f.read()
print("Количество символов:", len(text4))

Количество символов: 6276
Количество символов: 6322
Количество символов: 6352
Количество символов: 5610


In [4]:
import math
import re
from collections import Counter

texts = [text1, text2, text3, text4]
labels = ["text1", "text2", "text3", "text4"]

def tokenize(text):
    return re.findall(r"[а-яёa-z]+", text.lower())

docs = [tokenize(t) for t in texts]
N = len(docs)

vocab = sorted(set(word for doc in docs for word in doc))
df = {word: sum(1 for doc in docs if word in set(doc)) for word in vocab}

vectors = []
for doc in docs:
    counts = Counter(doc)
    doc_len = len(doc) if len(doc) > 0 else 1
    vec = []
    for word in vocab:
        tf = counts[word] / doc_len
        idf = math.log((N + 1) / (df[word] + 1)) + 1
        vec.append(tf * idf)
    vectors.append(vec)

def cosine_similarity(vec_a, vec_b):
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

sim_matrix = []
for i in range(N):
    row = []
    for j in range(N):
        row.append(cosine_similarity(vectors[i], vectors[j]))
    sim_matrix.append(row)

print("Матрица косинусной близости (TF-IDF):")
print("      " + "  ".join(labels))
for i, row in enumerate(sim_matrix):
    row_str = "  ".join(f"{value:.3f}" for value in row)
    print(f"{labels[i]}  {row_str}")

best_pair = None
best_score = -1.0
for i in range(N):
    for j in range(i + 1, N):
        if sim_matrix[i][j] > best_score:
            best_score = sim_matrix[i][j]
            best_pair = (i, j)

i, j = best_pair
print(f"\nБлиже всего друг к другу: {labels[i]} и {labels[j]} (cosine = {best_score:.3f})")

Матрица косинусной близости (TF-IDF):
      text1  text2  text3  text4
text1  1.000  0.281  0.312  0.272
text2  0.281  1.000  0.408  0.260
text3  0.312  0.408  1.000  0.298
text4  0.272  0.260  0.298  1.000

Ближе всего друг к другу: text2 и text3 (cosine = 0.408)


Комментарий к результату:

Тексты: text1 - отрывок из «Войны и мира» (художественная проза), text2 — статья с Хабра про ИИ в 2025 году (разговорно-технический стиль), text3 — статья Википедии про глобальное потепление (научный стиль), text4 - статья Википедии про головной мозг (тоже научный).

Ближайшая пара - text2 и text3 (cosine = 0.408), и это для меня неожиданно, потому что темы у них совершенно разные - технологии и климат. Я ожидала, что ближе окажутся text3 и text4, ведь это обе статьи из Википедии в одном стиле. Но наверное дело в том, что text2 и text3 оба очень насыщены статистикой и цифрами. TF-IDF не воспринимает их как стоп слова, потому что в художественном тексте (text1) такого нет, вот они и вносят большой вклад в сходство.

text3 и text4 всё равно довольно близки (0.298) - что логично, оба из Википедии.

text1 (Война и мир», как и ожидалось, дальше всего от остальных - там совсем другая лексика, диалоги, описания, никакой терминологии. Меньше всего похож на него text4 (0.272) - видимо потому что анатомические термины из статьи про мозг вообще не пересекаются с тем, что пишет Толстой.